# Week 12: Dijkstra's Algorithm — Weighted Graphs
## PHASE 5: Network Structures

*📚 Data Structures & Algorithms · ⏱️ 3 Hours · 👨‍🏫 Dr. Arif Solmaz*

## 🎯 Learning Objectives

1. Recall how BFS finds the shortest path in **unweighted** graphs
2. Understand why BFS fails when edges have different **weights**
3. Represent **weighted graphs** in Python
4. Understand Dijkstra’s algorithm: the **greedy** approach
5. Walk through Dijkstra’s algorithm **step by step**
6. Implement Dijkstra’s using `heapq` (priority queue)
7. Build a **mini route planner** for finding shortest paths between cities
8. Understand Dijkstra’s limitation: **no negative weights**

## 🎯 Core Mastery Connection

When edges have costs, BFS fails — it finds fewest hops, not cheapest paths. Dijkstra finds cheapest paths using a heap (from Week 08!) as a priority queue. This week connects two data structure choices: graphs for modeling networks, and heaps for efficient greedy selection. You will benchmark sparse vs dense graphs to see how edge count affects Dijkstra's performance.

---
## 🧭 Three-Hour Interactive Studio Plan

**Audience:** Mechatronics Engineering students  
**Weekly focus:** Week 12: Dijkstra's Algorithm — Weighted Graphs

**Professional lens:** planning least-cost routes for mobile robots and logistics systems.

| Time | Learning cycle |
|---|---|
| 00:00–00:10 | Launch question, prior-knowledge retrieval, outcomes |
| 00:10–00:50 | Concept cycle 1: explain → predict → test |
| 00:50–01:00 | Checkpoint 1, student questions, peer explanation |
| 01:00–01:10 | Break |
| 01:10–01:50 | Concept cycle 2: worked example → variation → discussion |
| 01:50–02:00 | Checkpoint 2 and misconception repair |
| 02:00–02:10 | Break |
| 02:10–02:40 | Core in-class practice with instructor circulation |
| 02:40–02:50 | Checkpoint 3: exam bridge and professional transfer |
| 02:50–03:00 | Open questions, summary, and exit ticket |

The official start and finish times are followed as published in the timetable. Ask questions at any point; the scheduled checkpoints guarantee additional question time. Checkpoints are private self-checks in this runtime—no identity, upload, homework, or instructor dashboard.


In [ ]:
# Run once. This pulse stays only in the current Colab runtime.
_studio_pulses = {}

def studio_pulse(number, response, minimum_words=8):
    words = str(response).strip().split()
    ready = len(words) >= minimum_words
    _studio_pulses[int(number)] = ready
    if ready:
        print(f"✅ Checkpoint {number}: explanation recorded locally ({len(words)} words).")
    else:
        print(f"🟡 Checkpoint {number}: explain your reasoning in at least {minimum_words} words, then retry.")
    print("Nothing is transmitted or stored for grading.")
    return ready

print("✅ Local studio checkpoints ready")

---
## 📦 Setup

Run this cell first to load the required packages.

In [ ]:
import matplotlib.pyplot as plt

import heapq
import random
import time

---
## Part 1: Review — BFS and Unweighted Shortest Paths

Last week we learned that **BFS** finds the shortest path in terms of **number of edges** (hops).

But what if edges have different **costs**?

```
Unweighted:               Weighted:
  A --- B --- D              A --2-- B --7-- D
  |           |              |               |
  C --------- E              C ------1------ E

BFS shortest A→D: A-B-D     Cheapest A→D: A-C-E-D (cost 1+1+1=3)
(2 hops)                     Not A-B-D (cost 2+7=9)!
```

**BFS doesn’t work for weighted graphs** because fewer edges doesn’t mean lower cost.

**Figure 1.1** — Why BFS fails on weighted graphs

In [ ]:
# A weighted graph where BFS gives the WRONG shortest path
#
#   A --10-- B
#   |        |
#   1        1
#   |        |
#   C --1--- D
#
# BFS from A to B: A → B (1 hop, but cost = 10!)
# Actual cheapest:  A → C → D → B (3 hops, but cost = 1+1+1 = 3!)

print("Weighted graph example:")
print("  A --10-- B")
print("  |        |")
print("  1        1")
print("  |        |")
print("  C --1--- D")
print()
print("BFS shortest path A → B: A → B (1 hop, cost = 10)")
print("Actual cheapest path:    A → C → D → B (3 hops, cost = 3)")
print()
print("❌ BFS picks the path with fewest edges, not lowest cost!")
print("✅ We need Dijkstra's algorithm for weighted graphs.")

---
## Part 2: Weighted Graph Representation

For weighted graphs, each edge has a **weight** (cost, distance, time, etc.).

We modify our adjacency list to store **(neighbor, weight)** tuples instead of just neighbors.

```
Unweighted:                    Weighted:
graph = {                      graph = {
    "A": ["B", "C"],              "A": [("B", 4), ("C", 1)],
    "B": ["A", "D"],              "B": [("A", 4), ("D", 3)],
    ...                            ...
}                              }
```

**Real-world analogy:** A road map where cities are nodes and the distances between them are edge weights.

**Figure 2.1** — Creating a weighted graph

In [ ]:
# Weighted graph as adjacency list with (neighbor, weight) tuples
#
#   A ---4--- B
#   |         |  \
#   1         3    2
#   |         |      \
#   C ---2--- D ---5--- E

weighted_graph = {
    "A": [("B", 4), ("C", 1)],
    "B": [("A", 4), ("D", 3), ("E", 2)],
    "C": [("A", 1), ("D", 2)],
    "D": [("B", 3), ("C", 2), ("E", 5)],
    "E": [("B", 2), ("D", 5)]
}

print("Weighted Graph (Adjacency List):")
for node, edges in weighted_graph.items():
    connections = [f"{neighbor}(cost:{weight})" for neighbor, weight in edges]
    print(f"  {node} → {', '.join(connections)}")

**Figure 2.2** — Helper function for building weighted graphs

In [ ]:
def add_weighted_edge(graph, node1, node2, weight):
    """Add an undirected weighted edge."""
    if node1 not in graph:
        graph[node1] = []
    if node2 not in graph:
        graph[node2] = []
    graph[node1].append((node2, weight))
    graph[node2].append((node1, weight))


# Build a city distance graph
cities = {}
add_weighted_edge(cities, "Istanbul", "Ankara", 450)
add_weighted_edge(cities, "Istanbul", "Bursa", 155)
add_weighted_edge(cities, "Bursa", "Eskisehir", 230)
add_weighted_edge(cities, "Eskisehir", "Ankara", 235)
add_weighted_edge(cities, "Ankara", "Konya", 260)
add_weighted_edge(cities, "Konya", "Antalya", 300)
add_weighted_edge(cities, "Ankara", "Antalya", 480)

print("City distances (km):")
for city, connections in cities.items():
    for neighbor, dist in connections:
        if city < neighbor:  # Print each edge once
            print(f"  {city} ↔ {neighbor}: {dist} km")

---
## Part 3: Dijkstra’s Algorithm — The Idea

Dijkstra’s algorithm finds the **shortest (cheapest) path** from a start node to all other nodes in a weighted graph.

**Key idea (Greedy):**
> Always process the **closest unvisited node** first.

**Step by step:**
1. Set the distance to the start node = 0, all others = infinity
2. Put the start node in a **priority queue** (min-heap)
3. While the priority queue is not empty:
   - Pop the node with the **smallest distance**
   - For each neighbor, calculate the **new distance** through the current node
   - If this new distance is **shorter** than the known distance, update it

**Real-world analogy:** Imagine you’re navigating with Google Maps. At each intersection, you always take the road that has the **lowest total distance so far**. You never go back to a place you’ve already fully explored.

| Property | Value |
|----------|-------|
| Strategy | Greedy (always pick cheapest next) |
| Data structure | Priority queue (min-heap) |
| Time complexity | O((V + E) log V) with a heap |
| Works with negative weights? | No! |

---
### ⏱️ Checkpoint 1 of 3 — Think · Pair · Explain

For **Week 12: Dijkstra's Algorithm — Weighted Graphs**, state the key invariant, operation cost, or decision rule in your own words.

First write a private prediction. Then explain it to a partner, revise it, and enter your final explanation below. Ask a question now if any step is unclear.


In [ ]:
checkpoint_1_response = ""  # write at least 8 words
studio_pulse(1, checkpoint_1_response)

---
## Part 4: Step-by-Step Walkthrough

Let’s trace through Dijkstra’s algorithm on a small graph.

**Figure 4.1** — Manual walkthrough of Dijkstra's algorithm

In [ ]:
# Graph:
#   A --4-- B
#   |       |  
#   1       3  
#   |       |  
#   C --2-- D --5-- E
#
# Find shortest path from A to all nodes

print("Dijkstra's Algorithm — Step by Step")
print("===================================\n")

print("Initial distances:")
print("  A=0, B=∞, C=∞, D=∞, E=∞")
print("  Priority Queue: [(0, A)]\n")

print("Step 1: Pop A (distance 0)")
print("  Check neighbor B: 0 + 4 = 4 < ∞ → update B to 4")
print("  Check neighbor C: 0 + 1 = 1 < ∞ → update C to 1")
print("  Distances: A=0, B=4, C=1, D=∞, E=∞")
print("  PQ: [(1, C), (4, B)]\n")

print("Step 2: Pop C (distance 1) — closest unvisited!")
print("  Check neighbor A: already visited")
print("  Check neighbor D: 1 + 2 = 3 < ∞ → update D to 3")
print("  Distances: A=0, B=4, C=1, D=3, E=∞")
print("  PQ: [(3, D), (4, B)]\n")

print("Step 3: Pop D (distance 3)")
print("  Check neighbor B: 3 + 3 = 6 > 4 → no update (B already has 4)")
print("  Check neighbor C: already visited")
print("  Check neighbor E: 3 + 5 = 8 < ∞ → update E to 8")
print("  Distances: A=0, B=4, C=1, D=3, E=8")
print("  PQ: [(4, B), (8, E)]\n")

print("Step 4: Pop B (distance 4)")
print("  Check neighbor A: already visited")
print("  Check neighbor D: already visited")
print("  Check neighbor E: 4 + 2 = 6 < 8 → update E to 6!")
print("  Distances: A=0, B=4, C=1, D=3, E=6")
print("  PQ: [(6, E)]\n")

print("Step 5: Pop E (distance 6)")
print("  All neighbors visited. Done!")
print()
print("Final shortest distances from A:")
print("  A=0, B=4, C=1, D=3, E=6")

---
## Part 5: Implementation Using heapq

We use Python’s `heapq` module as our priority queue. Remember from Week 08:
- `heapq.heappush(heap, item)` — add an item
- `heapq.heappop(heap)` — remove and return the **smallest** item

We store `(distance, node)` tuples so the heap always gives us the closest node.

**Figure 5.1** — Dijkstra's algorithm implementation

In [ ]:
import heapq

def dijkstra(graph, start):
    """
    Dijkstra's algorithm: find shortest distances from start to all nodes.
    Returns a dict of {node: shortest_distance}.
    """
    # Initialize all distances to infinity
    distances = {node: float('inf') for node in graph}
    distances[start] = 0
    
    # Priority queue: (distance, node)
    pq = [(0, start)]
    
    # Track visited nodes
    visited = set()
    
    while pq:
        # Get the node with the smallest distance
        current_dist, current_node = heapq.heappop(pq)
        
        # Skip if already visited
        if current_node in visited:
            continue
        visited.add(current_node)
        
        print(f"  Visit {current_node} (distance: {current_dist})")
        
        # Check all neighbors
        for neighbor, weight in graph[current_node]:
            if neighbor not in visited:
                new_dist = current_dist + weight
                if new_dist < distances[neighbor]:
                    distances[neighbor] = new_dist
                    heapq.heappush(pq, (new_dist, neighbor))
    
    return distances


# Test with our graph
graph = {
    "A": [("B", 4), ("C", 1)],
    "B": [("A", 4), ("D", 3), ("E", 2)],
    "C": [("A", 1), ("D", 2)],
    "D": [("B", 3), ("C", 2), ("E", 5)],
    "E": [("B", 2), ("D", 5)]
}

print("Running Dijkstra from A:")
distances = dijkstra(graph, "A")
print(f"\nShortest distances from A:")
for node, dist in sorted(distances.items()):
    print(f"  A → {node}: {dist}")

---
## Part 6: Dijkstra with Path Tracking

Knowing the shortest **distance** is useful, but we usually also want the actual **path**. We can track the path by remembering the **previous node** for each node.

**Figure 6.1** — Dijkstra with path reconstruction

In [ ]:
import heapq

def dijkstra_with_path(graph, start, end):
    """
    Find the shortest path and distance from start to end.
    Returns (distance, path_list).
    """
    distances = {node: float('inf') for node in graph}
    distances[start] = 0
    previous = {node: None for node in graph}  # Track the path
    
    pq = [(0, start)]
    visited = set()
    
    while pq:
        current_dist, current_node = heapq.heappop(pq)
        
        if current_node in visited:
            continue
        visited.add(current_node)
        
        # Found the destination — we can stop early!
        if current_node == end:
            break
        
        for neighbor, weight in graph[current_node]:
            if neighbor not in visited:
                new_dist = current_dist + weight
                if new_dist < distances[neighbor]:
                    distances[neighbor] = new_dist
                    previous[neighbor] = current_node  # Remember where we came from
                    heapq.heappush(pq, (new_dist, neighbor))
    
    # Reconstruct the path
    path = []
    node = end
    while node is not None:
        path.append(node)
        node = previous[node]
    path.reverse()
    
    return distances[end], path


# Test
dist, path = dijkstra_with_path(graph, "A", "E")
print(f"Shortest path A → E: {' → '.join(path)}")
print(f"Total cost: {dist}")

print()

dist2, path2 = dijkstra_with_path(graph, "C", "E")
print(f"Shortest path C → E: {' → '.join(path2)}")
print(f"Total cost: {dist2}")

---
## Part 7: Mini Route Planner — Finding Shortest Paths Between Cities

Let’s build a practical route planner using real-ish distances between Turkish cities.

**Figure 7.1** — City route planner

In [ ]:
import heapq

# Turkish cities with approximate road distances (km)
turkey_roads = {
    "Istanbul":  [("Ankara", 450), ("Bursa", 155), ("Edirne", 235)],
    "Ankara":    [("Istanbul", 450), ("Eskisehir", 235), ("Konya", 260), ("Antalya", 480)],
    "Bursa":     [("Istanbul", 155), ("Eskisehir", 230)],
    "Eskisehir": [("Bursa", 230), ("Ankara", 235), ("Afyon", 180)],
    "Konya":     [("Ankara", 260), ("Antalya", 300), ("Afyon", 230)],
    "Antalya":   [("Ankara", 480), ("Konya", 300)],
    "Edirne":    [("Istanbul", 235)],
    "Afyon":     [("Eskisehir", 180), ("Konya", 230)]
}


def find_route(graph, start, end):
    """Find the shortest route between two cities."""
    distances = {node: float('inf') for node in graph}
    distances[start] = 0
    previous = {node: None for node in graph}
    pq = [(0, start)]
    visited = set()
    
    while pq:
        current_dist, current_node = heapq.heappop(pq)
        if current_node in visited:
            continue
        visited.add(current_node)
        if current_node == end:
            break
        for neighbor, weight in graph[current_node]:
            if neighbor not in visited:
                new_dist = current_dist + weight
                if new_dist < distances[neighbor]:
                    distances[neighbor] = new_dist
                    previous[neighbor] = current_node
                    heapq.heappush(pq, (new_dist, neighbor))
    
    # Reconstruct path
    path = []
    node = end
    while node is not None:
        path.append(node)
        node = previous[node]
    path.reverse()
    
    return distances[end], path


# Find routes
routes_to_check = [
    ("Istanbul", "Antalya"),
    ("Edirne", "Konya"),
    ("Bursa", "Antalya"),
]

print("🗺️  Turkish City Route Planner")
print("=" * 45)
for start, end in routes_to_check:
    dist, path = find_route(turkey_roads, start, end)
    print(f"\n{start} → {end}:")
    print(f"  Route: {' → '.join(path)}")
    print(f"  Distance: {dist} km")

**Figure 7.2** — Comparing direct vs optimal routes

In [ ]:
import heapq

# Let's show ALL distances from Istanbul
def dijkstra_all(graph, start):
    """Find shortest distances from start to ALL nodes."""
    distances = {node: float('inf') for node in graph}
    distances[start] = 0
    previous = {node: None for node in graph}
    pq = [(0, start)]
    visited = set()
    
    while pq:
        current_dist, current_node = heapq.heappop(pq)
        if current_node in visited:
            continue
        visited.add(current_node)
        for neighbor, weight in graph[current_node]:
            if neighbor not in visited:
                new_dist = current_dist + weight
                if new_dist < distances[neighbor]:
                    distances[neighbor] = new_dist
                    previous[neighbor] = current_node
                    heapq.heappush(pq, (new_dist, neighbor))
    
    return distances, previous


def get_path(previous, end):
    path = []
    node = end
    while node is not None:
        path.append(node)
        node = previous[node]
    return list(reversed(path))


distances, previous = dijkstra_all(turkey_roads, "Istanbul")

print("All shortest distances from Istanbul:")
print("-" * 50)
for city in sorted(distances.keys()):
    if city == "Istanbul":
        continue
    path = get_path(previous, city)
    print(f"  → {city:12s}  {distances[city]:4d} km  ({' → '.join(path)})")

---
## Part 8: Understanding the Priority Queue

The priority queue (min-heap) is the key to Dijkstra’s efficiency. It ensures we always process the **cheapest available node** next.

Remember from Week 08: `heapq` gives us O(log n) push and pop operations.

**Figure 8.1** — How the priority queue changes during Dijkstra

In [ ]:
import heapq

def dijkstra_verbose(graph, start):
    """Dijkstra with detailed priority queue state."""
    distances = {node: float('inf') for node in graph}
    distances[start] = 0
    pq = [(0, start)]
    visited = set()
    step = 0
    
    while pq:
        step += 1
        current_dist, current_node = heapq.heappop(pq)
        
        if current_node in visited:
            print(f"  Step {step}: Skip {current_node} (already visited)")
            continue
        visited.add(current_node)
        
        print(f"  Step {step}: Pop {current_node} (dist={current_dist})")
        
        for neighbor, weight in graph[current_node]:
            if neighbor not in visited:
                new_dist = current_dist + weight
                if new_dist < distances[neighbor]:
                    old_dist = distances[neighbor]
                    distances[neighbor] = new_dist
                    heapq.heappush(pq, (new_dist, neighbor))
                    old_str = '∞' if old_dist == float('inf') else str(old_dist)
                    print(f"         Update {neighbor}: {old_str} → {new_dist}")
        
        pq_str = [(d, n) for d, n in sorted(pq)]
        print(f"         PQ: {pq_str}")
        print()
    
    return distances


# Small graph for clear visualization
small = {
    "A": [("B", 4), ("C", 1)],
    "B": [("A", 4), ("D", 3)],
    "C": [("A", 1), ("D", 2)],
    "D": [("B", 3), ("C", 2)]
}

print("Dijkstra step-by-step from A:")
print("=" * 45)
result = dijkstra_verbose(small, "A")
print(f"Final distances: {result}")

---
## Part 9: Dijkstra’s Limitations — No Negative Weights

Dijkstra’s algorithm does **not work** with negative edge weights!

**Why?** Because Dijkstra assumes that once a node is visited, its distance is final. A negative edge could reduce the distance to an already-visited node.

| Feature | Dijkstra |
|---------|----------|
| Positive weights | Works correctly |
| Zero weights | Works correctly |
| Negative weights | Does NOT work |
| Negative cycles | Does NOT work |

**Figure 9.1** — Demonstrating failure with negative weights

In [ ]:
import heapq

# Graph with a negative edge:
#   A --5-- B
#   |       |
#   2      -10  ← negative!
#   |       |
#   C --3-- D
#
# Dijkstra finds A→C = 2 (correct)
# But the true shortest A→C might go through the -10 edge!
# A→B→D→C = 5 + (-10) + 3 = -2 (cheaper than 2!)

negative_graph = {
    "A": [("B", 5), ("C", 2)],
    "B": [("A", 5), ("D", -10)],  # Negative edge!
    "C": [("A", 2), ("D", 3)],
    "D": [("B", -10), ("C", 3)]
}

def dijkstra_simple(graph, start):
    distances = {node: float('inf') for node in graph}
    distances[start] = 0
    pq = [(0, start)]
    visited = set()
    while pq:
        current_dist, current_node = heapq.heappop(pq)
        if current_node in visited:
            continue
        visited.add(current_node)
        for neighbor, weight in graph[current_node]:
            if neighbor not in visited:
                new_dist = current_dist + weight
                if new_dist < distances[neighbor]:
                    distances[neighbor] = new_dist
                    heapq.heappush(pq, (new_dist, neighbor))
    return distances

result = dijkstra_simple(negative_graph, "A")
print("Dijkstra's result (with negative edge):")
for node, dist in sorted(result.items()):
    print(f"  A → {node}: {dist}")

print()
print("Dijkstra says A → C = 2")
print("But A → B → D → C = 5 + (-10) + 3 = -2 (cheaper!)")
print()
print("❌ Dijkstra gives the WRONG answer with negative weights!")
print("   For negative weights, use the Bellman-Ford algorithm instead.")

---
### ⏱️ Checkpoint 2 of 3 — Think · Pair · Explain

Predict what happens when the input size doubles. Justify the trend with an operation count or complexity class—not timing alone.

First write a private prediction. Then explain it to a partner, revise it, and enter your final explanation below. Ask a question now if any step is unclear.


In [ ]:
checkpoint_2_response = ""  # write at least 8 words
studio_pulse(2, checkpoint_2_response)

---
## Part 10: Common Errors

**Figure 10.1** — Common error: forgetting to skip visited nodes

In [ ]:
import heapq

# ❌ Without checking visited, we process nodes multiple times
def dijkstra_broken(graph, start):
    distances = {node: float('inf') for node in graph}
    distances[start] = 0
    pq = [(0, start)]
    process_count = 0
    
    while pq:
        current_dist, current_node = heapq.heappop(pq)
        process_count += 1
        # ❌ No visited check — will process same node multiple times!
        for neighbor, weight in graph[current_node]:
            new_dist = current_dist + weight
            if new_dist < distances[neighbor]:
                distances[neighbor] = new_dist
                heapq.heappush(pq, (new_dist, neighbor))
    
    return distances, process_count


test_graph = {
    "A": [("B", 1), ("C", 4)],
    "B": [("A", 1), ("C", 2), ("D", 5)],
    "C": [("A", 4), ("B", 2), ("D", 1)],
    "D": [("B", 5), ("C", 1)]
}

result, count = dijkstra_broken(test_graph, "A")
print(f"❌ Without visited check: processed nodes {count} times")
print(f"   (With only {len(test_graph)} nodes, should be at most {len(test_graph)}!)")
print()
print("✅ Always skip already-visited nodes to avoid redundant work.")

**Figure 10.2** — Common error: using a regular list instead of heapq

In [ ]:
# ❌ Using a regular list and finding the minimum manually
# This works but is SLOW: O(V²) instead of O((V+E) log V)

def dijkstra_slow(graph, start):
    """Dijkstra without a heap — O(V²) time."""
    distances = {node: float('inf') for node in graph}
    distances[start] = 0
    visited = set()
    comparisons = 0
    
    while len(visited) < len(graph):
        # ❌ Find minimum by scanning ALL unvisited nodes — slow!
        min_dist = float('inf')
        min_node = None
        for node in graph:
            comparisons += 1
            if node not in visited and distances[node] < min_dist:
                min_dist = distances[node]
                min_node = node
        
        if min_node is None:
            break
        
        visited.add(min_node)
        for neighbor, weight in graph[min_node]:
            if neighbor not in visited:
                new_dist = min_dist + weight
                if new_dist < distances[neighbor]:
                    distances[neighbor] = new_dist
    
    return distances, comparisons


result, comps = dijkstra_slow(test_graph, "A")
print(f"Slow Dijkstra (no heap): {comps} comparisons to find minimums")
print(f"Results: {result}")
print()
print("✅ Use heapq for O(log n) min extraction instead of O(n) scanning.")

**Figure 10.3** — Common error: graph with missing nodes

In [ ]:
# ❌ If a neighbor is not in the graph dict, we get a KeyError
bad_graph = {
    "A": [("B", 3)],
    # "B" is missing as a key!
}

try:
    distances = {node: float('inf') for node in bad_graph}
    distances["A"] = 0
    # When processing A, we find neighbor B
    # But B is not in our distances dict!
    for neighbor, weight in bad_graph["A"]:
        if weight < distances[neighbor]:  # KeyError!
            pass
except KeyError as e:
    print(f"❌ KeyError: {e}")
    print("   Node B is a neighbor of A but not a key in the graph!")

print()

# ✅ Make sure every node that appears as a neighbor also has its own entry
good_graph = {
    "A": [("B", 3)],
    "B": [("A", 3)],  # ✅ B has its own entry
}
print("✅ Every node should have its own key in the adjacency list.")

---
## Part 11: Summary Table

| Algorithm | Graph Type | Finds | Data Structure | Time |
|-----------|-----------|-------|---------------|------|
| BFS | Unweighted | Shortest path (fewest edges) | Queue | O(V + E) |
| DFS | Any | Reachability, all paths | Stack | O(V + E) |
| Dijkstra | Weighted (non-negative) | Shortest path (lowest cost) | Priority Queue | O((V+E) log V) |

---
## 🌉 Bridge to Next Week

Congratulations! You’ve now learned the major graph algorithms covered in this introductory course:

| Week | Topic | Key Takeaway |
|------|-------|--------------|
| Week 10 | BST | Fast search & insert in tree structure |
| Week 11 | BFS & DFS | Traversing and searching graphs |
| Week 12 | Dijkstra | Finding cheapest paths in weighted graphs |

These algorithms form the foundation of many real-world systems: GPS navigation, social networks, routing protocols, and more.

**Next steps for further study:**
- Bellman-Ford algorithm (handles negative weights)
- A* algorithm (Dijkstra + heuristics for faster pathfinding)
- Minimum spanning trees (Prim’s and Kruskal’s algorithms)

---
## Part 12: Performance Benchmark — Dijkstra on Sparse vs Dense Graphs

> **🎯 Predict first, then measure. Does reality match your prediction?**
>
> Dijkstra's complexity is O((V + E) log V). A sparse graph has E proportional to V; a dense graph has E proportional to V^2. Before running, predict: how much slower will the dense graph be compared to the sparse one for n=1000?

How does Dijkstra's algorithm perform as graph size increases? And how does the number of edges (sparse vs dense) affect the running time? Let's find out by benchmarking on randomly generated weighted graphs.

In [ ]:
import time
import random
import heapq
import matplotlib.pyplot as plt

def generate_weighted_graph(n, avg_edges_per_node):
    """Generate a random undirected weighted graph."""
    graph = {i: [] for i in range(n)}
    total_edges = (n * avg_edges_per_node) // 2
    for _ in range(total_edges):
        u = random.randint(0, n - 1)
        v = random.randint(0, n - 1)
        if u != v:
            w = random.randint(1, 100)
            graph[u].append((v, w))
            graph[v].append((u, w))
    return graph

def dijkstra_bench(graph, start):
    """Dijkstra without print statements for benchmarking."""
    distances = {node: float('inf') for node in graph}
    distances[start] = 0
    pq = [(0, start)]
    visited = set()
    while pq:
        current_dist, current_node = heapq.heappop(pq)
        if current_node in visited:
            continue
        visited.add(current_node)
        for neighbor, weight in graph[current_node]:
            if neighbor not in visited:
                new_dist = current_dist + weight
                if new_dist < distances[neighbor]:
                    distances[neighbor] = new_dist
                    heapq.heappush(pq, (new_dist, neighbor))
    return distances

sizes = [50, 100, 200, 500, 1000]
num_runs = 5

sparse_times = []
dense_times = []

for n in sizes:
    # Sparse graph: ~3 edges per node
    times = []
    for _ in range(num_runs):
        g = generate_weighted_graph(n, avg_edges_per_node=3)
        start = time.time()
        dijkstra_bench(g, 0)
        times.append(time.time() - start)
    sparse_times.append(sum(times) / num_runs)

    # Dense graph: ~n/2 edges per node
    times = []
    for _ in range(num_runs):
        g = generate_weighted_graph(n, avg_edges_per_node=n // 2)
        start = time.time()
        dijkstra_bench(g, 0)
        times.append(time.time() - start)
    dense_times.append(sum(times) / num_runs)

    print(f"n={n:5d}: sparse={sparse_times[-1]:.5f}s  dense={dense_times[-1]:.5f}s")

plt.figure(figsize=(10, 6))
plt.plot(sizes, sparse_times, 'o-', label='Sparse (~3 edges/node)', linewidth=2)
plt.plot(sizes, dense_times, 's-', label='Dense (~n/2 edges/node)', linewidth=2)
plt.xlabel('Number of Nodes (n)')
plt.ylabel('Average Time (seconds)')
plt.title("Dijkstra's Algorithm: Sparse vs Dense Graphs")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

**Observations:**
- **Sparse graphs** (with ~3 edges per node) remain fast even as the number of nodes grows, since Dijkstra's complexity is O((V + E) log V) and E stays proportional to V.
- **Dense graphs** (with ~n/2 edges per node) grow much faster because the number of edges E grows as O(n^2). Each node has many neighbors to process and more entries get pushed into the priority queue.
- This benchmark highlights why graph density matters when choosing algorithms. For very dense graphs, a simpler O(V^2) Dijkstra implementation (without a heap) can sometimes be competitive, since the heap overhead becomes significant when E is close to V^2.

---
## 🎢 Exercises

### Easy Exercises

**EX1 (Easy):** Create a weighted graph for the following and print the adjacency list:
- A to B: cost 3
- A to C: cost 1
- B to D: cost 4
- C to D: cost 2
- D to E: cost 6

Expected Output:
```
A: [('B', 3), ('C', 1)]
B: [('A', 3), ('D', 4)]
C: [('A', 1), ('D', 2)]
D: [('B', 4), ('C', 2), ('E', 6)]
E: [('D', 6)]
```

<details><summary>💡 Hint</summary>
Use the add_weighted_edge helper or create the dictionary manually with (neighbor, weight) tuples.
</details>

In [ ]:
# ✏️ [EX1] Your code here


**EX2 (Easy):** Run Dijkstra’s algorithm on the graph from EX1, starting from node "A". Print the shortest distance to each node.

Expected Output:
```
A → A: 0
A → B: 3
A → C: 1
A → D: 3
A → E: 9
```

<details><summary>💡 Hint</summary>
Use the dijkstra function from Part 5. It returns a dictionary of shortest distances.
</details>

In [ ]:
# ✏️ [EX2] Your code here


**EX3 (Easy):** Find the shortest path from A to E using Dijkstra’s with path tracking. Print both the path and the total cost.

Expected Output:
```
Path: A → C → D → E
Cost: 9
```

<details><summary>💡 Hint</summary>
Use the dijkstra_with_path function from Part 6.
</details>

In [ ]:
# ✏️ [EX3] Your code here


**EX4 (Easy):** Add a direct edge from A to E with cost 15 to the graph from EX1. Run Dijkstra again from A. Does the shortest path to E change?

Expected Output:
```
Shortest distance A → E: 9
Path: A → C → D → E
The direct A-E edge (cost 15) is NOT the shortest!
```

<details><summary>💡 Hint</summary>
Add ("E", 15) to A's neighbor list and ("A", 15) to E's neighbor list. Then rerun Dijkstra.
</details>

In [ ]:
# ✏️ [EX4] Your code here


### Medium Exercises

**EX5 (Medium):** Write a function `total_weight(graph)` that calculates the total weight of all edges in an undirected weighted graph. Remember each edge appears twice in the adjacency list!

Test with: A-B(3), B-C(5), A-C(2)

Expected Output:
```
Total weight: 10
```

<details><summary>💡 Hint</summary>
Sum all weights in the adjacency list, then divide by 2 (since each edge is counted twice).
</details>

In [ ]:
# ✏️ [EX5] Your code here


**EX6 (Medium):** Write a function `cheapest_neighbor(graph, node)` that returns the neighbor with the lowest edge cost.

Test with the graph from EX1.

Expected Output:
```
Cheapest neighbor of A: C (cost 1)
Cheapest neighbor of D: C (cost 2)
```

<details><summary>💡 Hint</summary>
Use min() with a key function on the list of (neighbor, weight) tuples.
</details>

In [ ]:
# ✏️ [EX6] Your code here


---
### ⏱️ Checkpoint 3 of 3 — Think · Pair · Explain

Choose one completed core exercise. Explain why the algorithm is correct, its dominant cost, and one mechatronics situation where that cost matters.

First write a private prediction. Then explain it to a partner, revise it, and enter your final explanation below. Ask a question now if any step is unclear.


In [ ]:
checkpoint_3_response = ""  # write at least 8 words
studio_pulse(3, checkpoint_3_response)

---
## 🌟 Optional Extension

Exercises 7 and above are enrichment for remaining class time or independent curiosity. They are not homework and are not collected.


**EX7 (Medium):** Modify the `dijkstra_with_path` function to return ALL shortest paths from a start node to ALL other nodes (not just one destination). Return a dictionary of `{node: (distance, path)}`.

Expected Output:
```
A → A: 0 via [A]
A → B: 3 via [A, B]
A → C: 1 via [A, C]
A → D: 3 via [A, C, D]
A → E: 9 via [A, C, D, E]
```

<details><summary>💡 Hint</summary>
Remove the early exit when finding the destination. After the algorithm finishes, reconstruct paths for ALL nodes using the previous dictionary.
</details>

In [ ]:
# ✏️ [EX7] Your code here


**EX8 (Medium):** Create a delivery route planner. Given a weighted graph of locations and a list of delivery stops, find the total distance of visiting all stops **in order** using the shortest path between consecutive stops.

Graph edges: A-B(5), A-C(3), B-C(2), B-D(4), C-D(6), D-E(1)

Delivery stops: A → C → D → E

Expected Output:
```
A → C: 3 km
C → D: 6 km
D → E: 1 km
Total delivery distance: 10 km
```

<details><summary>💡 Hint</summary>
For each consecutive pair of stops, run dijkstra_with_path and sum up the distances.
</details>

In [ ]:
# ✏️ [EX8] Your code here


**EX9 (Medium):** Write a function `reachable_within(graph, start, max_cost)` that returns all nodes reachable from start with a total cost of at most `max_cost`.

Test with the EX1 graph, start="A", max_cost=3.

Expected Output:
```
Reachable from A within cost 3: ['A', 'B', 'C', 'D']
```

<details><summary>💡 Hint</summary>
Run Dijkstra from start, then filter the results to only include nodes with distance <= max_cost.
</details>

In [ ]:
# ✏️ [EX9] Your code here


**EX10 (Medium):** Compare the running time of BFS shortest path vs Dijkstra on an **unweighted** graph (where all edges have weight 1). Are the results the same? Use the `time` module to measure execution time.

Build a graph with at least 6 nodes.

Expected Output:
```
BFS path A → F: ['A', ...., 'F'], cost: X
Dijkstra path A → F: ['A', ...., 'F'], cost: X
Both find the same path: True
```

<details><summary>💡 Hint</summary>
Create an unweighted graph for BFS and the same graph with weight=1 on every edge for Dijkstra. Compare the results.
</details>

In [ ]:
# ✏️ [EX10] Your code here


### Challenge Exercises

**EX11 (Challenge):** Build a **flight cost optimizer**. Given a graph of cities with flight costs, find the cheapest route from a source to a destination with a maximum of `k` stops.

```python
flights = {
    "IST": [("ANK", 100), ("ADB", 400), ("AYT", 500)],
    "ANK": [("IST", 100), ("ADB", 200), ("AYT", 300)],
    "ADB": [("IST", 400), ("ANK", 200), ("AYT", 100)],
    "AYT": [("IST", 500), ("ANK", 300), ("ADB", 100)]
}
```

Find cheapest IST → AYT with max 1 stop.

Expected Output:
```
Cheapest IST → AYT (max 1 stop): 400
Route: IST → ANK → AYT
```

<details><summary>💡 Hint</summary>
Modify Dijkstra to track (cost, node, stops_remaining) in the priority queue. Only add neighbors if stops_remaining > 0.
</details>

In [ ]:
# ✏️ [EX11] Your code here


**EX12 (Challenge):** Implement a **network delay time** calculator. Given a weighted directed graph representing a network, find how long it takes for a signal sent from a start node to reach ALL other nodes. If any node is unreachable, return -1.

```python
network = {
    1: [(2, 3), (3, 5)],
    2: [(3, 1), (4, 6)],
    3: [(4, 2)],
    4: [],
    5: []   # Isolated node!
}
```

Expected Output:
```
Signal from node 1:
  → Node 2: 3 ms
  → Node 3: 4 ms
  → Node 4: 6 ms
  → Node 5: unreachable
Network delay time: -1 (not all nodes reachable)
```

<details><summary>💡 Hint</summary>
Run Dijkstra from the start node. The network delay time is the maximum distance among all reachable nodes. If any node has distance infinity, return -1.
</details>

In [ ]:
# ✏️ [EX12] Your code here
